# NammaRetina: reproducible GPU training
Select **Runtime → Change runtime type → T4 GPU** in Google Colab. This notebook downloads the versioned 3,662-image APTOS derivative, audits exact duplicates, trains with the corrected EfficientNet input range, selects on validation accuracy, and exports a production-compatible candidate. It cannot promise a particular score. The fixed test partition is evaluated only at the end. Dataset terms and source provenance remain applicable; the mirror is resized, not a second independent dataset.

Use a runtime version with Python 3.11 or 3.12. If the default is newer, choose an available past runtime in Runtime → Change runtime type; [official instructions](https://research.google.com/colaboratory/runtime-version-faq.html).


In [ ]:
from pathlib import Path
import subprocess, sys
repo = Path('/content/NammaRetina')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/The-ChandanKV/NammaRetina.git', str(repo)], check=True)
%cd /content/NammaRetina


In [ ]:
import sys
assert sys.version_info[:2] in {(3, 11), (3, 12)}, 'Select a past Colab runtime with Python 3.11/3.12 under Runtime > Change runtime type; see https://research.google.com/colaboratory/runtime-version-faq.html'
# Install before importing TensorFlow; restart the runtime if requested.
%pip install -r training/requirements.txt


In [ ]:
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'Enable a GPU runtime first'


In [ ]:
!python -m training.download_data


# Multi-dataset experiment targeting >90%
This is an experimental target, not a guaranteed result. Both datasets keep the original five DR grades. Validation selects the checkpoint; the test split must not be used to select training settings. The CPU baseline's test benchmark has already been inspected, so independent external confirmation remains necessary.


In [ ]:
!python -m training.download_data
!python -m training.download_data --dataset eyepacs224


In [ ]:
# Build a compatible starting checkpoint without inspecting the test set.
!python -m training.train --data-root training/data/aptos224 --run-dir training/outputs/gpu_start --mode frozen --require-gpu --no-test


In [ ]:
# At most 1,000 supplementary EyePACS images per grade; exact duplicates
# against all APTOS splits are excluded. The actual counts are recorded.
!python -m training.finetune --baseline training/outputs/gpu_start --run-dir training/outputs/gpu_multidata --extra-root training/data/eyepacs224 --extra-per-class 1000 --epochs 30 --batch-size 32 --learning-rate 1e-5 --class-weight-power 0 --train-last-layers 100 --no-test


In [ ]:
import json
from pathlib import Path
r=json.loads(Path('training/outputs/gpu_multidata/metrics.json').read_text())
print(json.dumps(r['validation'],indent=2))
print('This is validation performance, not a held-out test accuracy.')


## Final assessment
Only after selecting the final recipe by validation, run the next cell once. Do not repeat changes until the test score exceeds 90%. If it is below 90%, report that honestly; use a fresh independent holdout to assess further development.


In [ ]:
!python -m training.select_ensemble --baseline training/outputs/gpu_start --candidate training/outputs/gpu_multidata --run-dir training/outputs/gpu_final


In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')
source=Path('training/outputs/gpu_final')
dest=Path('/content/drive/MyDrive/NammaRetina/gpu_final')
dest.mkdir(parents=True,exist_ok=True)
for name in ['efficientnetb0_dr.keras','metrics.json','model.sha256','model_metadata.json']:
    shutil.copy2(source/name,dest/name)
print((source/'metrics.json').read_text())
print('Saved artifacts to',dest)


## Production
Review all five class recalls as well as overall accuracy. Upload the selected `.keras` artifact, configure Render `MODEL_URL` and `MODEL_SHA256`, and redeploy. Verify `/health` reports the correct loaded hash. A benchmark score above 90% would not establish clinical validation. See training/README.md for integration steps.
